<a href="https://colab.research.google.com/github/maabmusa7/BinX_Tech_Internship_Project_GROUP5/blob/ai%2Fwhisperx-pronunciation-scoring/ai-ml/stt-whisperx/char_features_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Install + imports
!pip install -q whisperx
import whisperx
import torch
import pandas as pd
import numpy as np
import re
from tqdm.auto import tqdm
from datasets import load_dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# 2. Load dataset
dataset = load_dataset("mispeech/speechocean762")
train_set = dataset["train"]
test_set = dataset["test"]

# 3. Load alignment model
align_model, align_metadata = whisperx.load_align_model(language_code="en", device=device)

# 4. Helper function
def normalize_word(word):
    return re.sub(r"[^a-z']", "", word.lower())

## Resume Session

In [ ]:
from datasets import load_dataset

dataset = load_dataset("mispeech/speechocean762")

train_set = dataset["train"]
test_set = dataset["test"]

print("Train:", len(train_set))
print("Test:", len(test_set))

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/333M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/312M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2500 [00:00<?, ? examples/s]

Train: 2500
Test: 2500


In [ ]:
phone_rows = []

for i in range(len(train_set)):
    sample = train_set[i]

    for word in sample["words"]:

        phones = word["phones"]
        phone_accuracies = word["phones-accuracy"]

        for phone, phone_accuracy in zip(phones, phone_accuracies):

            phone_rows.append({
                "sample_id": i,
                "word": word["text"],
                "phone": phone,
                "phone_accuracy": phone_accuracy,
                "word_accuracy": word["accuracy"]
            })

phone_df = pd.DataFrame(phone_rows)

print("Total phone examples:", len(phone_df))

print("\nPhone accuracy distribution:")
print(
    phone_df["phone_accuracy"]
    .value_counts()
    .sort_index()
)

phone_df.head(20)

Total phone examples: 47076

Phone accuracy distribution:
phone_accuracy
0.0     1344
0.2       44
0.4      603
0.6       85
0.8      488
1.0      155
1.2      813
1.4      426
1.6     2416
1.8     2995
2.0    37707
Name: count, dtype: int64


,sample_id,word,phone,phone_accuracy,word_accuracy
0,0,WE,W,2.0,10
1,0,WE,IY0,2.0,10
2,0,CALL,K,2.0,10
3,0,CALL,AO0,1.8,10
4,0,CALL,L,1.8,10
5,0,IT,IH0,2.0,10
6,0,IT,T,2.0,10
7,0,BEAR,B,2.0,6
8,0,BEAR,EH0,1.0,6
9,0,BEAR,R,1.0,6


In [ ]:
# Create phone-level summary for each word
word_phone_summary = (
    phone_df
    .groupby(["sample_id", "word", "word_accuracy"])
    .agg(
        mean_phone_accuracy=("phone_accuracy", "mean"),
        min_phone_accuracy=("phone_accuracy", "min"),
        num_phones=("phone", "count")
    )
    .reset_index()
)

print("Total word examples:", len(word_phone_summary))

word_phone_summary.head(20)

Total word examples: 15501


,sample_id,word,word_accuracy,mean_phone_accuracy,min_phone_accuracy,num_phones
0,0,BEAR,6,1.333333,1.0,3
1,0,CALL,10,1.866667,1.8,3
2,0,IT,10,2.000000,2.0,2
3,0,WE,10,2.000000,2.0,2
4,1,FIVE,10,1.933333,1.8,3
5,1,ONE,10,2.000000,2.0,3
6,1,THREE,8,1.733333,1.2,3
7,1,ZERO,8,1.700000,1.4,4
8,2,SEVEN,10,2.000000,2.0,4
9,2,THREE,10,1.933333,1.8,3


In [ ]:
print(
    word_phone_summary[
        ["word_accuracy", "mean_phone_accuracy", "min_phone_accuracy"]
    ].corr()
)

                     word_accuracy  mean_phone_accuracy  min_phone_accuracy
word_accuracy             1.000000             0.863773            0.908087
mean_phone_accuracy       0.863773             1.000000            0.906445
min_phone_accuracy        0.908087             0.906445            1.000000


In [ ]:
def align_reference_text_with_chars(sample):

    # Get audio
    audio_samples = sample["audio"].get_all_samples()

    audio = audio_samples.data.numpy().squeeze()
    sample_rate = audio_samples.sample_rate

    # Audio duration
    duration = len(audio) / sample_rate

    # Use correct reference sentence
    reference_segments = [
        {
            "start": 0.0,
            "end": duration,
            "text": sample["text"]
        }
    ]

    # Run alignment and also return character-level scores
    aligned_result = whisperx.align(
        reference_segments,
        align_model,
        align_metadata,
        audio,
        device,
        return_char_alignments=True
    )

    return aligned_result

In [ ]:
sample = train_set[0]

char_result = align_reference_text_with_chars(sample)

print(char_result["segments"][0]["chars"])

[{'char': 'W', 'start': 0.665, 'end': 0.786, 'score': 0.99}, {'char': 'E', 'start': 0.786, 'end': 0.947, 'score': 0.843}, {'char': ' ', 'start': 0.947, 'end': 1.048, 'score': 0.899}, {'char': 'C', 'start': 1.048, 'end': 1.149, 'score': 0.998}, {'char': 'A', 'start': 1.149, 'end': 1.169, 'score': 0.945}, {'char': 'L', 'start': 1.169, 'end': 1.25, 'score': 0.891}, {'char': 'L', 'start': 1.25, 'end': 1.31, 'score': 0.63}, {'char': ' ', 'start': 1.31, 'end': 1.391, 'score': 0.788}, {'char': 'I', 'start': 1.391, 'end': 1.431, 'score': 0.976}, {'char': 'T', 'start': 1.431, 'end': 1.471, 'score': 0.99}, {'char': ' ', 'start': 1.471, 'end': 1.653, 'score': 0.964}, {'char': 'B', 'start': 1.653, 'end': 1.774, 'score': 0.992}, {'char': 'E', 'start': 1.774, 'end': 1.814, 'score': 0.654}, {'char': 'A', 'start': 1.814, 'end': 1.875, 'score': 0.661}, {'char': 'R', 'start': 1.875, 'end': 1.895, 'score': 0.382}]


In [ ]:
def extract_char_features(aligned_result):
    char_words = []

    # Get character alignment
    chars = aligned_result["segments"][0]["chars"]

    current_chars = []

    for item in chars:

        # Space means the current word is finished
        if item["char"] == " ":

            if current_chars:
                scores = [
                    c["score"]
                    for c in current_chars
                    if c.get("score") is not None
                ]

                if scores:
                    char_words.append({
                        "word": "".join(c["char"] for c in current_chars),

                        # Average character score
                        "mean_char_score": np.mean(scores),

                        # Lowest character score
                        "min_char_score": np.min(scores),

                        # Variation between character scores
                        "std_char_score": np.std(scores),

                        # Difference between highest and lowest score
                        "char_score_range": np.max(scores) - np.min(scores),

                        # Number of characters in the word
                        "num_chars": len(scores)
                    })

            current_chars = []

        else:
            current_chars.append(item)

    # Add the last word
    if current_chars:
        scores = [
            c["score"]
            for c in current_chars
            if c.get("score") is not None
        ]

        if scores:
            char_words.append({
                "word": "".join(c["char"] for c in current_chars),
                "mean_char_score": np.mean(scores),
                "min_char_score": np.min(scores),
                "std_char_score": np.std(scores),
                "char_score_range": np.max(scores) - np.min(scores),
                "num_chars": len(scores)
            })

    return char_words

In [ ]:
from tqdm.auto import tqdm

train_char_rows = []

skipped = 0
errors = 0

# Run character-level analysis on the full training set
for i in tqdm(range(len(train_set))):

    try:
        sample = train_set[i]

        # Character-level forced alignment
        aligned_result = align_reference_text_with_chars(sample)

        # Extract character features for every word
        char_words = extract_char_features(aligned_result)

        # Human-labelled words
        human_words = sample["words"]

        # Normalize words before comparing
        aligned_names = [
            normalize_word(w["word"])
            for w in char_words
        ]

        human_names = [
            normalize_word(w["text"])
            for w in human_words
        ]

        # Skip the sentence if the words do not match exactly
        if aligned_names != human_names:
            skipped += 1
            continue

        # Save features + ground truth
        for word_index, (char_word, human_word) in enumerate(
            zip(char_words, human_words)
        ):

            train_char_rows.append({
                "sample_id": i,
                "word_index": word_index,
                "word": human_word["text"],

                # Ground truth
                "human_accuracy": human_word["accuracy"],
                "pronunciation_issue": int(
                    human_word["accuracy"] <= 6
                ),

                # Character-level features
                "mean_char_score": char_word["mean_char_score"],
                "min_char_score": char_word["min_char_score"],
                "std_char_score": char_word["std_char_score"],
                "char_score_range": char_word["char_score_range"],
                "num_chars": char_word["num_chars"]
            })

    except Exception as e:
        errors += 1

        # Print only first few errors
        if errors <= 3:
            print(f"Error in sample {i}: {e}")


train_char_df = pd.DataFrame(train_char_rows)

print("\nUsable words:", len(train_char_df))
print("Skipped sentences:", skipped)
print("Errors:", errors)

  0%|          | 0/2500 [00:00<?, ?it/s]


Usable words: 15835
Skipped sentences: 2
Errors: 0


In [ ]:
train_char_df.to_csv(
    "/content/drive/MyDrive/train_char_features.csv",
    index=False
)

print("Saved!")

Saved!


In [ ]:
train_char_df.groupby("pronunciation_issue")[
    [
        "mean_char_score",
        "min_char_score",
        "std_char_score",
        "char_score_range"
    ]
].mean()

,mean_char_score,min_char_score,std_char_score,char_score_range
pronunciation_issue,,,,
0,0.792893,0.604905,0.132652,0.321228
1,0.657054,0.327041,0.221056,0.571538


In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

# Features produced by WhisperX
features = [
    "mean_char_score",
    "min_char_score",
    "std_char_score",
    "char_score_range"
]

X = train_char_df[features]

# Ground truth:
# 0 = acceptable pronunciation
# 1 = pronunciation issue
y = train_char_df["pronunciation_issue"]

# Keep words from the same sentence together
groups = train_char_df["sample_id"]

# Split full training data into:
# 80% development train
# 20% validation
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_val = X.iloc[val_idx]

y_train = y.iloc[train_idx]
y_val = y.iloc[val_idx]

print("Training words:", len(X_train))
print("Validation words:", len(X_val))

print("\nTraining issues:")
print(y_train.value_counts())

print("\nValidation issues:")
print(y_val.value_counts())

Training words: 12651
Validation words: 3184

Training issues:
pronunciation_issue
0    11397
1     1254
Name: count, dtype: int64

Validation issues:
pronunciation_issue
0    2888
1     296
Name: count, dtype: int64


In [ ]:
# Baseline V2:
# combine multiple character-level features

model_v2 = Pipeline([
    ("scaler", StandardScaler()),

    # Balanced because pronunciation issues are the minority class
    ("classifier", LogisticRegression(
        class_weight="balanced",
        max_iter=1000
    ))
])

# Train only on development training data
model_v2.fit(X_train, y_train)

# Predict on validation data
val_predictions = model_v2.predict(X_val)

precision = precision_score(y_val, val_predictions)
recall = recall_score(y_val, val_predictions)
f1 = f1_score(y_val, val_predictions)

print("Validation Precision:", round(precision, 4))
print("Validation Recall:", round(recall, 4))
print("Validation F1:", round(f1, 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, val_predictions))

Validation Precision: 0.2212
Validation Recall: 0.6622
Validation F1: 0.3316

Confusion Matrix:
[[2198  690]
 [ 100  196]]


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

# Get probability that each validation word has a pronunciation issue
val_probabilities = model_v2.predict_proba(X_val)[:, 1]

threshold_results = []

# Try different thresholds
for threshold in np.arange(0.05, 0.96, 0.01):

    predictions = (
        val_probabilities >= threshold
    ).astype(int)

    threshold_results.append({
        "threshold": threshold,
        "precision": precision_score(
            y_val,
            predictions,
            zero_division=0
        ),
        "recall": recall_score(
            y_val,
            predictions,
            zero_division=0
        ),
        "f1": f1_score(
            y_val,
            predictions,
            zero_division=0
        )
    })


v2_threshold_df = pd.DataFrame(threshold_results)

# Show thresholds with highest F1
v2_threshold_df.sort_values(
    "f1",
    ascending=False
).head(10)

,threshold,precision,recall,f1
57,0.62,0.267016,0.516892,0.352129
52,0.57,0.251095,0.581081,0.350663
54,0.59,0.257552,0.547297,0.350270
67,0.72,0.317549,0.385135,0.348092
53,0.58,0.252280,0.560811,0.348008
56,0.61,0.260504,0.523649,0.347924
51,0.56,0.246132,0.591216,0.347567
50,0.55,0.243537,0.604730,0.347236
58,0.63,0.264973,0.493243,0.344746
66,0.71,0.306878,0.391892,0.344214


In [ ]:
best_v2_row = v2_threshold_df.loc[
    v2_threshold_df["f1"].idxmax()
]

print("Best V2 threshold:", round(best_v2_row["threshold"], 2))
print("Precision:", round(best_v2_row["precision"], 4))
print("Recall:", round(best_v2_row["recall"], 4))
print("F1:", round(best_v2_row["f1"], 4))

Best V2 threshold: 0.62
Precision: 0.267
Recall: 0.5169
F1: 0.3521


In [ ]:
test_char_rows = []

skipped = 0
errors = 0

# Run character-level analysis on the full test set
for i in tqdm(range(len(test_set))):

    try:
        sample = test_set[i]

        # Character-level forced alignment
        aligned_result = align_reference_text_with_chars(sample)

        # Extract character features
        char_words = extract_char_features(aligned_result)

        # Human-labelled words
        human_words = sample["words"]

        # Normalize words before comparing
        aligned_names = [
            normalize_word(w["word"])
            for w in char_words
        ]

        human_names = [
            normalize_word(w["text"])
            for w in human_words
        ]

        # Skip if words do not match exactly
        if aligned_names != human_names:
            skipped += 1
            continue

        # Save features + ground truth
        for word_index, (char_word, human_word) in enumerate(
            zip(char_words, human_words)
        ):

            test_char_rows.append({
                "sample_id": i,
                "word_index": word_index,
                "word": human_word["text"],

                # Ground truth
                "human_accuracy": human_word["accuracy"],
                "pronunciation_issue": int(
                    human_word["accuracy"] <= 6
                ),

                # Character-level features
                "mean_char_score": char_word["mean_char_score"],
                "min_char_score": char_word["min_char_score"],
                "std_char_score": char_word["std_char_score"],
                "char_score_range": char_word["char_score_range"],
                "num_chars": char_word["num_chars"]
            })

    except Exception as e:
        errors += 1

        if errors <= 3:
            print(f"Error in sample {i}: {e}")


test_char_df = pd.DataFrame(test_char_rows)

print("\nUsable words:", len(test_char_df))
print("Skipped sentences:", skipped)
print("Errors:", errors)

  0%|          | 0/2500 [00:00<?, ?it/s]


Usable words: 15928
Skipped sentences: 5
Errors: 0


In [ ]:
test_char_df.to_csv(
    "/content/drive/MyDrive/test_char_features.csv",
    index=False
)

print("Test character features saved!")

Test character features saved!


In [ ]:
# Features used by V2
features = [
    "mean_char_score",
    "min_char_score",
    "std_char_score",
    "char_score_range"
]

X_full_train = train_char_df[features]
y_full_train = train_char_df["pronunciation_issue"]

X_test = test_char_df[features]
y_test = test_char_df["pronunciation_issue"]


# Train the final V2 model using all training data
final_model_v2 = Pipeline([
    ("scaler", StandardScaler()),

    ("classifier", LogisticRegression(
        class_weight="balanced",
        max_iter=1000
    ))
])

final_model_v2.fit(X_full_train, y_full_train)

print("Final V2 model trained!")

Final V2 model trained!


In [ ]:
# Fixed threshold selected from validation
V2_THRESHOLD = 0.62

# Probability of pronunciation issue
test_probabilities_v2 = final_model_v2.predict_proba(X_test)[:, 1]

# Final predictions
test_pred_v2 = (
    test_probabilities_v2 >= V2_THRESHOLD
).astype(int)


# Evaluation
precision_v2 = precision_score(
    y_test,
    test_pred_v2,
    zero_division=0
)

recall_v2 = recall_score(
    y_test,
    test_pred_v2,
    zero_division=0
)

f1_v2 = f1_score(
    y_test,
    test_pred_v2,
    zero_division=0
)


print("V2 Test Precision:", round(precision_v2, 4))
print("V2 Test Recall:", round(recall_v2, 4))
print("V2 Test F1:", round(f1_v2, 4))

print("\nV2 Test Confusion Matrix:")
print(confusion_matrix(y_test, test_pred_v2))

V2 Test Precision: 0.2207
V2 Test Recall: 0.4996
V2 Test F1: 0.3062

V2 Test Confusion Matrix:
[[12382  2263]
 [  642   641]]


In [ ]:
# V1 baseline:
# one alignment/mean score + fixed threshold

V1_THRESHOLD = 0.73

test_pred_v1 = (
    test_char_df["mean_char_score"] < V1_THRESHOLD
).astype(int)


precision_v1 = precision_score(
    y_test,
    test_pred_v1,
    zero_division=0
)

recall_v1 = recall_score(
    y_test,
    test_pred_v1,
    zero_division=0
)

f1_v1 = f1_score(
    y_test,
    test_pred_v1,
    zero_division=0
)


print("V1 Test Precision:", round(precision_v1, 4))
print("V1 Test Recall:", round(recall_v1, 4))
print("V1 Test F1:", round(f1_v1, 4))

print("\nV1 Test Confusion Matrix:")
print(confusion_matrix(y_test, test_pred_v1))

V1 Test Precision: 0.1636
V1 Test Recall: 0.5721
V1 Test F1: 0.2544

V1 Test Confusion Matrix:
[[10892  3753]
 [  549   734]]


## Summary: WhisperX Pronunciation Detection — V1 → V2

### Approach 1 (V1): Raw alignment score + fixed threshold
- Used WhisperX forced alignment to get one score per word (via `align_reference_text`)
- Ground truth label: `human_accuracy <= 6` → pronunciation issue (1), else acceptable (0)
- Searched thresholds 0.00–1.00 on the **train set only**, selected by best F1
- **Best threshold: 0.73**

| Split | Precision | Recall | F1 |
|---|---|---|---|
| Train | 0.2065 | 0.5848 | 0.3052 |
| Test  | 0.1638 | 0.5694 | 0.2544 |

Conclusion: the raw alignment score carries some signal, but a single threshold on it alone is not reliable enough.

### Phone-level exploration (insight, not a feature source)
- Used the dataset's own human-annotated phone-level accuracy (not from WhisperX)
- Correlation with word-level accuracy: **0.86–0.91**
- Confirms phone-level detail is highly informative — but this comes from human annotation, so it can't be used as a live input feature without a separate phoneme-recognition/GOP pipeline (future work).

### Approach 2 (V2): Character-level features + lightweight classifier
- Extracted 4 features per word from WhisperX's character-level alignment: `mean_char_score`, `min_char_score`, `std_char_score`, `char_score_range`
- Trained a **Logistic Regression** (`class_weight="balanced"`) on top — WhisperX itself stays frozen, only this small classifier is trained
- Split by `sample_id` (GroupShuffleSplit) to avoid word leakage across sentences from the same recording
- Threshold on predicted probability tuned on validation set: **0.62**

| Split | Precision | Recall | F1 |
|---|---|---|---|
| Validation | 0.2212 | 0.6622 | 0.3316 |
| Test (final)| 0.2207 | 0.4996 | 0.3062 |

### V1 vs V2 — same test set

| Version | Precision | Recall | F1 |
|---|---|---|---|
| V1 (raw score, threshold=0.73) | 0.1636 | 0.5721 | 0.2544 |
| V2 (char features + classifier, threshold=0.62) | 0.2207 | 0.4996 | **0.3062** |

**V2 improves F1 by ~20% over V1**, mainly by improving precision (fewer false alarms) at a similar recall level.

### Why we're stopping here
- Only ~10% of words are true pronunciation issues (severe class imbalance, confirmed in EDA) — this caps how high precision/recall can realistically get with a lightweight approach.
- WhisperX is an ASR model, not purpose-built for pronunciation scoring — some ceiling on performance is expected.
- Per mentor guidance, fine-tuning was out of scope; the priority was a reliable, well-evaluated frozen-model baseline within the time available.
- V2 is documented as the current best baseline, with phone-level scoring (GOP-style) flagged as the clear next step if time allows.

### What's next
Wrap this into a single callable function (`score_pronunciation(audio_bytes, transcript)`) so the AI service can call it directly, per the team's integration design (STT → parallel [LLM reply, pronunciation scorer] → combined response).